## 说明
Semantic Kernel 生产环境容错机制演示：服务降级与故障转移。这段代码演示了AI Agent在生产环境中的容错机制，特别展示了当主要服务不可用时如何自动切换到备份服务。这是生产级AI应用的关键特性，确保系统在部分组件失败时仍能提供基本服务。

### 核心演示内容
1. **服务降级机制**
    - 主要航班查询服务(`get_flight_times`)故意返回错误（HTTP 404）
    - 代理自动检测到错误并切换到备份服务(`get_flight_times_backup`)
    - 用户不会感知到服务故障，体验保持流畅
2. **生产级健壮性设计**
    - 明确的错误处理流程
    - 备份方案的无缝切换
    - 用户友好的错误沟通（不暴露技术细节）
3. **多工具调用策略**
    - 定义主工具和备份工具
    - 实现工具调用的优先级逻辑
    - 保持用户体验一致性

### 关键概念详解（针对Python小白）
#### 为什么需要容错机制？

想象一下你正在用手机订机票：
- 如果航空公司网站暂时打不开，你会怎么办？
- 好的订票系统会告诉你"正在尝试其他方式查询"，而不是直接显示错误

AI Agent也是一样：
- 当它依赖的服务(如航班查询)暂时不可用时
- 不能直接告诉用户"出错了，请重试"
- 而应该尝试其他方法完成任务

In [1]:
# 导入必要的库
import json  # 用于处理JSON数据格式
import os    # 用于操作系统相关功能，如读取环境变量

# 从typing模块导入Annotated类型，用于给函数参数添加描述信息
from typing import Annotated

# 从dotenv导入load_dotenv，用于加载.env文件中的环境变量
from dotenv import load_dotenv

# 从openai导入AsyncOpenAI，用于异步调用OpenAI API
from openai import AsyncOpenAI

# 从semantic_kernel导入关键组件
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion  # OpenAI聊天完成服务
from semantic_kernel.contents import (  # 不同类型的内容对象
    FunctionCallContent,     # 表示函数调用的内容
    FunctionResultContent,   # 表示函数调用结果的内容
    StreamingTextContent     # 表示流式文本响应的内容
)
from semantic_kernel.agents import (  # Agent相关组件
    ChatCompletionAgent,     # 基于聊天完成的Agent
    ChatHistoryAgentThread   # 用于维护对话历史的线程
)
from semantic_kernel.functions import kernel_function  # 用于定义可被Agent调用的函数


In [2]:
# 定义目的地插件类 - 包含代理可以调用的所有工具
class DestinationsPlugin:
    """提供旅行相关信息的插件，包含目的地列表和航班时间查询功能
    
    注意：这个类中的方法会被AI Agent当作"工具"来调用
    """

    @kernel_function(description="Provides a list of vacation destinations.")
    def get_destinations(self) -> Annotated[str, "Returns the specials from the menu."]:
        """获取可用的度假目的地列表
        
        Returns:
            str: 格式化的目的地列表字符串
        """
        return """
        Barcelona, Spain
        Paris, France
        Berlin, Germany
        Tokyo, Japan
        New York, USA
        """

    @kernel_function(description="Provides available flight times for a destination.")
    def get_flight_times(
        self, destination: Annotated[str, "The destination to check flight times for."]
    ) -> Annotated[str, "Returns flight times for the specified destination."]:
        """主航班查询服务 - 故意返回错误来模拟服务故障
        
        这是生产环境容错机制的关键部分：
        - 在真实场景中，这个服务可能因为各种原因失败
        - 我们故意让它返回错误，以测试故障转移机制
        
        Args:
            destination: 目的地名称
            
        Returns:
            str: 故意返回的HTTP 404错误信息
        """
        # 故意返回错误，模拟服务不可用的情况
        # Return HTTP ERROR 404
        return "HTTP ERROR 404: Flight times service is currently unavailable."

    @kernel_function(description="Backup function that provides available flight times for a destination.")
    def get_flight_times_backup(
        self, destination: Annotated[str, "The destination to check flight times for."]
    ) -> Annotated[str, "Returns flight times for the specified destination."]:
        """备份航班查询服务 - 当主服务失败时使用
        
        这是容错机制的关键部分：
        - 当主服务不可用时，自动切换到此服务
        - 提供与主服务相同的功能，但可能数据来源不同
        
        Args:
            destination: 目的地名称
            
        Returns:
            str: 格式化的航班时间信息
        """
        # 预定义的航班时间数据（模拟备份数据源）
        flight_times = {
            "Barcelona": ["08:30 AM", "02:15 PM", "10:45 PM"],
            "Paris": ["06:45 AM", "12:30 PM", "07:15 PM"],
            "Berlin": ["07:20 AM", "01:45 PM", "09:30 PM"],
            "Tokyo": ["11:00 AM", "05:30 PM", "11:55 PM"],
            "New York": ["05:15 AM", "03:00 PM", "08:45 PM"]
        }

        # Extract just the city name from input that might contain country
        # 从可能包含国家的输入中提取城市名
        city = destination.split(',')[0].strip()
        
        # 查找并返回航班时间
        if city in flight_times:
            times = ", ".join(flight_times[city])
            return f"Flight times for {city}: {times}"
        else:
            return f"No flight information available for {city}."

In [3]:
load_dotenv()
# 使用通义大模型，作为客户端
# model_name="qwen-max"
# client = AsyncOpenAI(
#     api_key=os.environ.get("DASHSCOPE_API_KEY"), 
#     base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
# )

# 使用GPT大模型，作为客户端
model_name = "gpt-4o-mini"
client = AsyncOpenAI(
    api_key=os.environ["GITHUB_TOKEN"],
    base_url="https://models.inference.ai.azure.com/"
)

# 创建AI服务对象，作为Semantic Kernel与Qwen-Max模型通信的桥梁
chat_completion_service = OpenAIChatCompletion(
    # 指定要使用的模型ID（这里是通义千问的qwen-max）
    ai_model_id=model_name,
    # 传入之前创建的AsyncOpenAI客户端
    async_client=client,
)

# chat_completion_service = OpenAIChatCompletion(
#     ai_model_id="gpt-4o-mini",
#     # async_client=client,
#     api_key=os.getenv("OPENAI_API_KEY"),
# )


In [4]:
# 定义Agent名称和指令
# 指令的中文翻译如下：
# 您是航班预订代理，提供有关可用航班的信息并在被询问时提供旅行活动建议。
# 旅行活动建议应针对客户、地点和停留时间。

# 您可以使用以下工具来帮助用户规划行程：
# 1. get_destinations：返回用户可以选择的度假目的地列表。
# 2. get_flight_times：提供特定目的地的可用航班时间。
# 3. get_flight_times_backup：当主服务中断时提供可用航班时间的备份功能。

# 您协助用户的流程：
# - 当用户查询航班预订时，使用 get_flight_times 预订前往他们所选目的地的最早航班。
# - 如果 get_flight_times 返回错误消息，请立即使用具有相同目的地参数的 get_flight_times_backup 来检索航班信息。
# - 由于您无法访问预订系统，请勿要求继续预订，只需假设您已预订航班即可。
# - 利用任何过去的对话历史记录来了解用户偏好，并在提供航班和活动建议时考虑这些偏好。如果建议是基于用户偏好的，请务必明确说明提出此建议的原因。

# 指南：
# - 使用工具时使用准确的目的地名称（巴塞罗那、巴黎、柏林、东京、纽约）
# - 以乐于助人和热情的方式回应旅行的可能性
# - 始终寻求反馈，以确保您的建议符合用户的期望
# - 当请求超出你的能力范围时，承认
# - 为了获得更好的格式，始终以列表格式显示航班时间
# - 提出任何限时建议时，请思考时间框架是否合理。如果不合理，请再次回复。
# - 如果航班时间服务中断，请告知用户您正在使用备用航班数据，同时保持积极的语气。

# 您的目标是通过了解用户的偏好并提供定制建议来帮助他们有效地探索度假选择并做出明智的旅行决定。
AGENT_NAME = "TravelAgent"
AGENT_INSTRUCTIONS = """ \
"You are Flight Booking Agent that provides information about available flights and gives travel activity suggestions when asked.
Travel activity suggestions should be specific to customer, location and amount of time at location.

You have access to the following tools to help users plan their trips:
1. get_destinations: Returns a list of available vacation destinations that users can choose from.
2. get_flight_times: Provides available flight times for specific destinations.
3. get_flight_times_backup: Backup function that provides available flight times when the primary service is down.

Your process for assisting users:
- When users inquire about flight booking, book the earliest flight available for the destination they choose using get_flight_times.
- If get_flight_times returns an error message, immediately use get_flight_times_backup with the same destination parameter to retrieve flight information.
- Since you do not have access to a booking system, DO NOT ask to proceed with booking, just assume you have booked the flight.
- Use any past conversation history to understand user preferences and consider them when making suggestions on flights and activities. When making a suggestion, be very clear on why you are making this suggestion if based on a user preference.

Guidelines:
- Use the exact destination names when using tools (Barcelona, Paris, Berlin, Tokyo, New York)
- Respond in a helpful and enthusiastic manner about travel possibilities
- Always seek feedback to ensure your suggestions meet the user's expectations
- Acknowledge when a request falls outside your capabilities
- For better formatting, always display flight times in a list format
- When giving any timed suggestions, reflect if the time frames are reasonable. Respond again if not.
- If the flight times service is down, inform the user that you're using backup flight data while maintaining a positive tone.

Your goal is to help users explore vacation options efficiently and make informed travel decisions by understanding their preferences and providing tailored recommendations.
"""
# Create the agent
# 创建AI代理实例
agent = ChatCompletionAgent(
    service=chat_completion_service,  # 使用配置好的聊天服务
    plugins=[DestinationsPlugin()],  # 注册目的地插件（包含所有工具）
    name=AGENT_NAME,  # 代理名称
    instructions=AGENT_INSTRUCTIONS,  # 代理指令，定义了行为规则
)

In [5]:
from IPython.display import display, HTML

# 为演示准备的用户输入
# 这里只有一个简单的请求，但足以展示故障转移机制
user_inputs = [
    "Book me a flight to Barcelona",
]

# 创建线程来保存对话历史
# 如果未提供线程，将创建新线程并在初始响应中返回
thread: ChatHistoryAgentThread | None = None

async def main():
    """主函数：演示生产环境中的容错机制
    
    本函数展示了：
    1. 如何处理流式响应和函数调用
    2. 当主服务失败时如何自动切换到备份服务
    3. 如何向用户呈现友好的错误处理体验
    
    重要：这是生产级AI Agent的关键特性
    """
    global thread
    
    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []

        # 用于重建流式函数调用的缓冲区
        current_function_name = None
        argument_buffer = ""

        # 调用代理的流式响应接口
        # 注意：这里传递了thread参数以维护对话历史
        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread  # 更新对话线程
            agent_name = response.name  # 获取代理名称
            content_items = list(response.items)  # 获取响应内容项

            # 处理每个内容项
            for item in content_items:
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # Accumulate arguments (streamed in chunks)
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                elif isinstance(item, FunctionResultContent):
                    # Finalize any pending function call before showing result
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # leave as raw string

                        function_calls.append(f"Calling function: {current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\nFunction Result:\n\n{item.result}")
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()



---

**免责声明**：  
本文档使用AI翻译服务[Co-op Translator](https://github.com/Azure/co-op-translator)进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于关键信息，建议使用专业人工翻译。我们对因使用此翻译而产生的任何误解或误读不承担责任。
